# Model Development

In [ ]:
# Stdlib
import os

# Core scientific stack
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Scikit-learn
from sklearn.model_selection import GroupKFold, RandomizedSearchCV
from sklearn.metrics import (
    classification_report,
    r2_score,
    mean_squared_error,
)
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor

# XGBoost
from xgboost import XGBClassifier, XGBRegressor

# SciPy
from scipy.stats import randint, uniform, pearsonr, spearmanr
from scipy.special import expit  # sigmoid

# Utilities
import joblib
from scipy.signal import medfilt

import joblib
from xgboost import XGBClassifier
from scipy.stats import randint, uniform
from imblearn.under_sampling import TomekLinks

In [ ]:
# Path to your saved pickle
pkl_path = "data/processed/zscores_df.pkl"

zscores_df = pd.read_pickle(pkl_path)

In [ ]:
zscores_df[["Clip_Ratio","Label"]]

In [ ]:
zscores_df["Record"].unique()

In [ ]:
zscores_df["Record"].unique()

In [ ]:
zscores_df.columns

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

# Extract data and drop NaNs
x = zscores_df["Clip_Ratio"].dropna().values

# Summary statistics
mean_val = np.mean(x)
median_val = np.median(x)
max_val = np.max(x)
std_val = np.std(x, ddof=1)

# X range for Gaussian
x_range = np.linspace(x.min(), x.max(), 500)
pdf = norm.pdf(x_range, mean_val, std_val)

# Plot
plt.figure(figsize=(10, 5))

# Histogram (density)
plt.hist(x, bins=30, density=True, alpha=0.6, edgecolor="black")

# Gaussian fit
plt.plot(x_range, pdf, linewidth=2, label=f"Gaussian fit (μ={mean_val:.3f}, σ={std_val:.3f})")

# Reference lines
plt.axvline(mean_val, linestyle="--", linewidth=2, label=f"Mean = {mean_val:.3f}")
plt.axvline(median_val, linestyle=":", linewidth=2, label=f"Median = {median_val:.3f}")
plt.axvline(max_val, linestyle="-.", linewidth=2, label=f"Max = {max_val:.3f}")

# Labels
plt.xlabel("Clip_Ratio")
plt.ylabel("Density")
plt.title("Distribution of Clip_Ratio with Gaussian Fit")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
zscores_df["Record"].unique()

In [ ]:
# Identify records starting with Case or FID
mask_records = zscores_df["Record"].str.startswith(("Case", "FID"))

# Create an index mask to drop first 50 rows per matching Record
drop_idx = (
    zscores_df[mask_records]
    .groupby("Record")
    .head(50)
    .index
)

# Drop those rows
zscores_df = zscores_df.drop(index=drop_idx).reset_index(drop=True)

In [ ]:
zscores_df["Record"].unique()

## Classification Models

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)
from imblearn.under_sampling import TomekLinks
from scipy.stats import randint
from collections import defaultdict

# ===============================
# CONFIG
# ===============================
random_state = 42
n_folds = 5

def get_sampling_rate(record_id: str) -> int:
    return 250 if record_id.lower().startswith("cu") else 240

# ===============================
# LOAD DATA
# ===============================
df = zscores_df.copy()

# -------------------------------
# Drop excluded subjects
# -------------------------------
excluded_subjects = [
    'cu01','cu02','cu12','cu14','cu21','cu30','cu31','cu33','cu34','cu35'
]
df = df[~df['Record'].isin(excluded_subjects)].copy()

# -------------------------------
# Feature list
# -------------------------------
features = [
    'QT_Interval_Z','TMV_Score_Z','Mean_HR_Z','QRS_Global_Z',"Clip_Ratio",
    'T_Flatness_Z','TWAmp_Std_Z','TWAmp_CV_Z','TMV_Global_Z',
    'QRS_Duration_Z','QRS_Area_Z','QRS_Skewness_Z','avail',
    'ST_Deviation_Mean_Z','ST_Slope_Mean_Z',
    'AC_ECG_Peak_Z','AC_ECG_Lag_Sec_Z','AC_ECG_MeanAroundPeak_Z',
    'AC_RR_Peak_Z','AC_RR_Lag_Beats_Z','AC_RR_MeanAroundPeak_Z'
]

df = df.dropna(subset=features + ['VTAC_Label', 'Record', 'ECG_Raw', 'Start'])

# ===============================
# ML INPUTS
# ===============================
X = df[features].values
y = df['VTAC_Label'].values
groups = df['Record'].astype(str).values

# ===============================
# BUILD CUSTOM FOLDS (SUBJECT-LEVEL)
# ===============================
subjects = np.unique(groups)

fid_subjects  = sorted([s for s in subjects if s.startswith("FID")])
case_subjects = sorted([s for s in subjects if s.startswith("Case_")])
cu_subjects   = sorted([s for s in subjects if s.lower().startswith("cu")])

fold_subjects = defaultdict(list)

# -------------------------------
# Distribute FID subjects evenly
# -------------------------------
for i, fid in enumerate(fid_subjects):
    fold_subjects[i % n_folds].append(fid)

# -------------------------------
# Distribute Case subjects evenly  ✅ FIX
# -------------------------------
for i, case in enumerate(case_subjects):
    fold_subjects[i % n_folds].append(case)

# -------------------------------
# Distribute CU subjects evenly
# -------------------------------
for i, cu in enumerate(cu_subjects):
    fold_subjects[i % n_folds].append(cu)

# -------------------------------
# Convert to index folds
# -------------------------------
fold_indices = []

for fold_id in range(n_folds):

    test_subjects = fold_subjects[fold_id]
    test_mask = np.isin(groups, test_subjects)

    test_idx  = np.where(test_mask)[0]
    train_idx = np.where(~test_mask)[0]

    fold_indices.append((train_idx, test_idx))

    # Optional sanity print
    print(f"\nFold {fold_id + 1}")
    print(" Test subjects:", test_subjects)
    print("  FID :", [s for s in test_subjects if s.startswith("FID")])
    print("  Case:", [s for s in test_subjects if s.startswith("Case_")])
    print("  CU  :", [s for s in test_subjects if s.lower().startswith("cu")])


# ===============================
# METRIC CONTAINERS
# ===============================
train_metrics = {"accuracy": [], "precision": [], "recall": [], "f1": []}
test_metrics  = {"accuracy": [], "precision": [], "recall": [], "f1": []}

# ===============================
# TRAINING LOOP
# ===============================
feature_importances = np.zeros(len(features))

for fold_num, (train_idx, test_idx) in enumerate(fold_indices, start=1):

    print(f"\n================ Fold {fold_num} =================")

    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    train_df = df.iloc[train_idx]
    test_df  = df.iloc[test_idx]
    test_records = test_df['Record'].unique()

    # -------------------------------
    # Tomek Links (TRAIN ONLY)
    # -------------------------------
    tomek = TomekLinks()
    X_train_clean, y_train_clean = tomek.fit_resample(X_train, y_train)
    print(f"Tomek removed {len(y_train) - len(y_train_clean)} samples")

    # -------------------------------
    # RandomizedSearchCV
    # -------------------------------
    param_dist = {
        'n_estimators': [50, 100, 200, 300],
        'max_depth': [3, 5, 7],
        'min_samples_split': randint(2, 10),
        'min_samples_leaf': randint(1, 5)
    }

    base_model = RandomForestClassifier(random_state=random_state)

    search = RandomizedSearchCV(
        base_model,
        param_distributions=param_dist,
        n_iter=50,
        scoring='f1',
        cv=3,
        n_jobs=-1,
        random_state=random_state
    )

    search.fit(X_train_clean, y_train_clean)
    model = search.best_estimator_

    # -------------------------------
    # Predictions
    # -------------------------------
    preds_train = model.predict(X_train_clean)
    preds_test  = model.predict(X_test)
    probs_test  = model.predict_proba(X_test)[:, 1]

    print("\nTrain Classification Report:")
    print(classification_report(y_train_clean, preds_train))

    print("Test Classification Report:")
    print(classification_report(y_test, preds_test))

    # -------------------------------
    # Collect metrics
    # -------------------------------
    train_metrics["accuracy"].append(accuracy_score(y_train_clean, preds_train))
    train_metrics["precision"].append(precision_score(y_train_clean, preds_train, zero_division=0))
    train_metrics["recall"].append(recall_score(y_train_clean, preds_train, zero_division=0))
    train_metrics["f1"].append(f1_score(y_train_clean, preds_train, zero_division=0))

    test_metrics["accuracy"].append(accuracy_score(y_test, preds_test))
    test_metrics["precision"].append(precision_score(y_test, preds_test, zero_division=0))
    test_metrics["recall"].append(recall_score(y_test, preds_test, zero_division=0))
    test_metrics["f1"].append(f1_score(y_test, preds_test, zero_division=0))

    feature_importances += model.feature_importances_

    # ===============================
    # SUBJECT-LEVEL PLOTS (UNCHANGED)
    # ===============================
    for subject_id in test_records:

        sr = get_sampling_rate(subject_id)
        subject_df = test_df[test_df['Record'] == subject_id].sort_values('Start')

        X_sub = subject_df[features].values
        subject_df['Prob_VTAC'] = model.predict_proba(X_sub)[:, 1]

        fig, axs = plt.subplots(2, 1, figsize=(14, 6), sharex=True)

        for _, row in subject_df.iterrows():
            ecg = row['ECG_Raw']
            if isinstance(ecg, list) and len(ecg) > 0:
                t0 = row['Start'] / sr
                t1 = t0 + 30
                t = np.linspace(t0, t1, len(ecg))
                axs[0].plot(t, ecg, color='black', alpha=0.4)

        vtac_rows = subject_df[subject_df['VTAC_Label'] == 1]
        if not vtac_rows.empty:
            vtac_start = vtac_rows['Start'].min() / sr
            vtac_end   = vtac_rows['Start'].max() / sr
            axs[0].axvline(vtac_start, color='red', linestyle='--')
            axs[0].axvline(vtac_end,   color='red', linestyle='--')
            axs[0].axvline(vtac_start - 60, color='blue', linestyle='--')

        axs[0].set_title(f"Raw ECG + VTAC Markers | {subject_id}")
        axs[0].set_ylabel("ECG")

        axs[1].plot(subject_df['Start'] / sr, subject_df['VTAC_Label'],
                    label="True", color='blue', alpha=0.6)
        axs[1].plot(subject_df['Start'] / sr, subject_df['Prob_VTAC'],
                    label="Predicted Prob", color='orange')

        axs[1].legend()
        axs[1].set_ylim(0, 1.05)
        axs[1].set_ylabel("P(VTAC)")
        axs[1].set_xlabel("Time (s)")

        plt.tight_layout()
        plt.show()

# ===============================
# FEATURE IMPORTANCE (AVERAGED)
# ===============================
feature_importances /= n_folds
sorted_idx = np.argsort(feature_importances)[::-1]

plt.figure(figsize=(10, 5))
plt.bar(range(len(features)), feature_importances[sorted_idx])
plt.xticks(range(len(features)),
           [features[i] for i in sorted_idx],
           rotation=45, ha='right')
plt.title("Average Feature Importance (Random Forest + Tomek)")
plt.ylabel("Importance")
plt.tight_layout()
plt.show()

# ===============================
# MEAN PERFORMANCE ACROSS FOLDS
# ===============================
print("\n==============================")
print("MEAN PERFORMANCE ACROSS FOLDS")
print("==============================")

def summarize_metrics(name, metrics):
    print(f"\n{name}")
    for k, v in metrics.items():
        v = np.array(v)
        print(f"  {k.capitalize():9s}: {v.mean():.3f} ± {v.std():.3f}")

summarize_metrics("TRAIN", train_metrics)
summarize_metrics("TEST", test_metrics)

In [ ]:
# --- Update ML inputs ---
X = df[features].values
y = df['VTAC_Label'].values

# -------------------------------
# Apply Tomek Links (FINAL TRAIN)
# -------------------------------
tomek = TomekLinks()
X_clean, y_clean = tomek.fit_resample(X, y)

print(f"Tomek removed {len(y) - len(y_clean)} samples from full dataset")


print("\n--- Training Final Random Forest Model (with Tomek) ---")

final_param_dist = {
    'n_estimators': [50, 100, 200, 300],
    'max_depth': [3, 5, 7],
    'min_samples_split': randint(2, 10),
    'min_samples_leaf': randint(1, 5)
}

final_base_model = RandomForestClassifier(random_state=42)

final_search = RandomizedSearchCV(
    final_base_model,
    param_distributions=final_param_dist,
    n_iter=50,
    cv=3,
    scoring='f1',
    random_state=42,
    n_jobs=-1
)

final_search.fit(X_clean, y_clean)
final_rf_model = final_search.best_estimator_

os.makedirs("model", exist_ok=True)

rf_model_path = os.path.join("model", "random_forest_vtac_model_binary.joblib")
joblib.dump(final_rf_model, rf_model_path)

print(f"✅ Final RF model saved to: {rf_model_path}")

print("\n--- Training Final XGBoost Model (with Tomek) ---")

final_param_dist = {
    'n_estimators': [50, 100, 200, 300],
    'max_depth': [3, 5, 7],
    'learning_rate': uniform(0.01, 0.2),
    'subsample': uniform(0.6, 0.4),
    'colsample_bytree': uniform(0.6, 0.4),
    'gamma': uniform(0, 5),
    'reg_alpha': uniform(0, 1),
    'reg_lambda': uniform(1, 2)
}

final_base_model = XGBClassifier(
    eval_metric='logloss',
    random_state=42
)

final_search = RandomizedSearchCV(
    final_base_model,
    param_distributions=final_param_dist,
    n_iter=50,
    cv=3,
    scoring='f1',
    random_state=42,
    n_jobs=-1
)

final_search.fit(X_clean, y_clean)
final_xgb_model = final_search.best_estimator_

xgb_model_path = os.path.join("model", "xgboost_vtac_model_binary.joblib")
joblib.dump(final_xgb_model, xgb_model_path)

print(f"✅ Final XGBoost model saved to: {xgb_model_path}")


## Regression Models

### XGBoosting Regression

In [ ]:
# ===============================
# CONFIG
# ===============================
random_state = 42
n_folds = 5
sigmoid_seconds = 300

def get_sampling_rate(record_id):
    return 250 if str(record_id).lower().startswith("cu") else 240

# ===============================
# LOAD DATA
# ===============================
df = zscores_df.copy()
df = df[~df["Record"].isin(excluded_subjects)].copy()

# ===============================
# REGRESSION LABEL (300s ramp)
# ===============================
df["VTAC_Label_Regression"] = 0.0

for record_id in df["Record"].unique():

    fs = get_sampling_rate(record_id)
    sigmoid_samples = int(sigmoid_seconds * fs)

    subj_idx = df["Record"] == record_id
    subj_df = df.loc[subj_idx]

    vtac_rows = subj_df[subj_df["VTAC_Label"] == 1]
    if vtac_rows.empty:
        continue

    vtac_start = vtac_rows["Start"].min()
    start_min = max(0, vtac_start - sigmoid_samples)

    pre_mask = (
        subj_idx &
        (df["Start"] >= start_min) &
        (df["Start"] < vtac_start)
    )

    i = vtac_start - df.loc[pre_mask, "Start"].values
    t = -i / fs
    t_norm = (t + sigmoid_seconds) / sigmoid_seconds

    df.loc[pre_mask, "VTAC_Label_Regression"] = t_norm ** 6
    df.loc[subj_idx & (df["VTAC_Label"] == 1), "VTAC_Label_Regression"] = 1.0

# ===============================
# ML INPUTS
# ===============================
df = df.dropna(
    subset=features + ["VTAC_Label_Regression", "Record", "ECG_Raw", "Start"]
)

X = df[features].values
y = df["VTAC_Label_Regression"].values
groups = df["Record"].astype(str).values

# ===============================
# BUILD BALANCED SUBJECT FOLDS
# ===============================
subjects = np.unique(groups)

fid_subjects  = sorted([s for s in subjects if s.startswith("FID")])
case_subjects = sorted([s for s in subjects if s.startswith("Case_")])
cu_subjects   = sorted([s for s in subjects if s.lower().startswith("cu")])

fold_subjects = defaultdict(list)

for i, fid in enumerate(fid_subjects):
    fold_subjects[i % n_folds].append(fid)

for i, case in enumerate(case_subjects):
    fold_subjects[i % n_folds].append(case)

for i, cu in enumerate(cu_subjects):
    fold_subjects[i % n_folds].append(cu)

fold_indices = []

for fold_id in range(n_folds):

    test_subjects = fold_subjects[fold_id]
    test_mask = np.isin(groups, test_subjects)

    test_idx  = np.where(test_mask)[0]
    train_idx = np.where(~test_mask)[0]

    fold_indices.append((train_idx, test_idx))

    print(f"\nFold {fold_id + 1}")
    print(" Test subjects:", test_subjects)
    print("  FID :", [s for s in test_subjects if s.startswith("FID")])
    print("  Case:", [s for s in test_subjects if s.startswith("Case_")])
    print("  CU  :", [s for s in test_subjects if s.lower().startswith("cu")])

# ===============================
# METRIC CONTAINERS
# ===============================
train_rmse_folds = []
test_rmse_folds  = []

all_preds_train, all_true_train = [], []
all_preds_test,  all_true_test  = [], []

feature_importances = np.zeros(len(features))

# ===============================
# TRAINING LOOP
# ===============================
for fold_num, (train_idx, test_idx) in enumerate(fold_indices, start=1):

    print(f"\n================ Fold {fold_num} ================")

    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    # -------------------------------
    # XGBoost Randomized Search
    # -------------------------------
    param_dist = {
        "n_estimators": [100, 200, 300],
        "max_depth": [3, 5, 7],
        "learning_rate": uniform(0.01, 0.2),
        "subsample": uniform(0.6, 0.4),
        "colsample_bytree": uniform(0.6, 0.4),
        "gamma": uniform(0, 5),
        "reg_alpha": uniform(0, 1),
        "reg_lambda": uniform(1, 2),
    }

    base_model = XGBRegressor(
        objective="reg:squarederror",
        random_state=random_state,
        verbosity=0,
        n_jobs=-1,
    )

    search = RandomizedSearchCV(
        base_model,
        param_distributions=param_dist,
        n_iter=50,
        cv=3,
        scoring="neg_mean_squared_error",
        random_state=random_state,
        n_jobs=-1,
    )

    search.fit(X_train, y_train)
    model = search.best_estimator_

    # -------------------------------
    # Predictions
    # -------------------------------
    preds_train = medfilt(model.predict(X_train), kernel_size=5)
    preds_test  = medfilt(model.predict(X_test),  kernel_size=5)

    rmse_train = np.sqrt(mean_squared_error(y_train, preds_train))
    rmse_test  = np.sqrt(mean_squared_error(y_test, preds_test))

    print(f"RMSE (Train/Test): {rmse_train:.4f} / {rmse_test:.4f}")

    train_rmse_folds.append(rmse_train)
    test_rmse_folds.append(rmse_test)

    all_preds_train.extend(preds_train)
    all_true_train.extend(y_train)

    all_preds_test.extend(preds_test)
    all_true_test.extend(y_test)

    feature_importances += model.feature_importances_

    # ===============================
    # SUBJECT-LEVEL PLOTS
    # ===============================
    test_records = np.unique(groups[test_idx])

    for subject_id in test_records:

        sr = get_sampling_rate(subject_id)

        subject_df = df.iloc[test_idx]
        subject_df = subject_df[subject_df["Record"] == subject_id]
        subject_df = subject_df.sort_values("Start")

        if subject_df.empty:
            continue

        X_sub = subject_df[features].values
        subject_df["Prediction"] = model.predict(X_sub)

        fig, axs = plt.subplots(2, 1, figsize=(14, 6), sharex=True)

        # --- Raw ECG ---
        for _, row in subject_df.iterrows():
            ecg = row["ECG_Raw"]
            if isinstance(ecg, list) and len(ecg) > 0:
                t = np.linspace(
                    row["Start"] / sr,
                    (row["Start"] + 30 * sr) / sr,
                    len(ecg),
                )
                axs[0].plot(t, ecg, color="black", alpha=0.35)

        # --- VTAC markers ---
        vtac_rows = subject_df[subject_df["VTAC_Label"] == 1]
        if not vtac_rows.empty:
            vtac_start = vtac_rows["Start"].min() / sr
            vtac_end   = vtac_rows["Start"].max() / sr

            axs[0].axvline(vtac_start, color="red", linestyle="--")
            axs[0].axvline(vtac_end,   color="red", linestyle="--")
            axs[0].axvline(vtac_start - sigmoid_seconds, color="blue", linestyle="--")

        axs[0].set_title(f"Raw ECG + VTAC | {subject_id}")
        axs[0].set_ylabel("ECG")

        # --- Regression ---
        axs[1].plot(
            subject_df["Start"] / sr,
            subject_df["VTAC_Label_Regression"],
            label="True",
            color="blue",
            alpha=0.6,
        )

        axs[1].plot(
            subject_df["Start"] / sr,
            subject_df["Prediction"],
            label="Predicted",
            color="orange",
        )

        axs[1].set_ylim(0, 1.1)
        axs[1].set_ylabel("VTAC Risk")
        axs[1].set_xlabel("Time (s)")
        axs[1].legend()

        plt.tight_layout()
        plt.show()

# ===============================
# FINAL METRICS
# ===============================
rmse_pooled_train = np.sqrt(
    mean_squared_error(all_true_train, all_preds_train)
)
rmse_pooled_test = np.sqrt(
    mean_squared_error(all_true_test, all_preds_test)
)

print("\n==============================")
print("CROSS-VALIDATION PERFORMANCE")
print("==============================")
print(f"Train RMSE: {np.mean(train_rmse_folds):.4f} ± {np.std(train_rmse_folds):.4f}")
print(f"Test  RMSE: {np.mean(test_rmse_folds):.4f} ± {np.std(test_rmse_folds):.4f}")

print("\n==============================")
print("POOLED PERFORMANCE")
print("==============================")
print(f"Pooled Train RMSE: {rmse_pooled_train:.4f}")
print(f"Pooled Test  RMSE: {rmse_pooled_test:.4f}")

# ===============================
# FEATURE IMPORTANCE
# ===============================
feature_importances /= n_folds
sorted_idx = np.argsort(feature_importances)[::-1]

plt.figure(figsize=(10, 5))
plt.bar(range(len(features)), feature_importances[sorted_idx])
plt.xticks(
    range(len(features)),
    [features[i] for i in sorted_idx],
    rotation=45,
    ha="right",
)
plt.title("XGBoost – Average Feature Importance")
plt.ylabel("Importance")
plt.tight_layout()
plt.show()


### Random Forest Regression

In [ ]:
# ===============================
# IMPORTS
# ===============================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from collections import defaultdict
from scipy.signal import medfilt
from scipy.stats import randint
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_squared_error

# ===============================
# CONFIG
# ===============================
random_state = 42
n_folds = 5
sigmoid_seconds = 300

def get_sampling_rate(record_id):
    return 250 if str(record_id).lower().startswith("cu") else 240

# ===============================
# LOAD DATA
# ===============================
df = zscores_df.copy()
df = df[~df["Record"].isin(excluded_subjects)].copy()

# ===============================
# REGRESSION LABEL (300s ramp)
# ===============================
df["VTAC_Label_Regression"] = 0.0

for record_id in df["Record"].unique():

    fs = get_sampling_rate(record_id)
    sigmoid_samples = int(sigmoid_seconds * fs)

    subj_idx = df["Record"] == record_id
    subj_df = df.loc[subj_idx]

    vtac_rows = subj_df[subj_df["VTAC_Label"] == 1]
    if vtac_rows.empty:
        continue

    vtac_start = vtac_rows["Start"].min()
    start_min = max(0, vtac_start - sigmoid_samples)

    pre_mask = (
        subj_idx &
        (df["Start"] >= start_min) &
        (df["Start"] < vtac_start)
    )

    i = vtac_start - df.loc[pre_mask, "Start"].values
    t = -i / fs
    t_norm = (t + sigmoid_seconds) / sigmoid_seconds

    df.loc[pre_mask, "VTAC_Label_Regression"] = t_norm ** 6
    df.loc[subj_idx & (df["VTAC_Label"] == 1), "VTAC_Label_Regression"] = 1.0

# ===============================
# ML INPUTS
# ===============================
df = df.dropna(
    subset=features + ["VTAC_Label_Regression", "Record", "ECG_Raw", "Start"]
)

X = df[features].values
y = df["VTAC_Label_Regression"].values
groups = df["Record"].astype(str).values

# ===============================
# BUILD BALANCED SUBJECT FOLDS
# ===============================
subjects = np.unique(groups)

fid_subjects  = sorted([s for s in subjects if s.startswith("FID")])
case_subjects = sorted([s for s in subjects if s.startswith("Case_")])
cu_subjects   = sorted([s for s in subjects if s.lower().startswith("cu")])

fold_subjects = defaultdict(list)

for i, fid in enumerate(fid_subjects):
    fold_subjects[i % n_folds].append(fid)

for i, case in enumerate(case_subjects):
    fold_subjects[i % n_folds].append(case)

for i, cu in enumerate(cu_subjects):
    fold_subjects[i % n_folds].append(cu)

fold_indices = []

for fold_id in range(n_folds):

    test_subjects = fold_subjects[fold_id]
    test_mask = np.isin(groups, test_subjects)

    test_idx  = np.where(test_mask)[0]
    train_idx = np.where(~test_mask)[0]

    fold_indices.append((train_idx, test_idx))

    print(f"\nFold {fold_id + 1}")
    print(" Test subjects:", test_subjects)
    print("  FID :", [s for s in test_subjects if s.startswith("FID")])
    print("  Case:", [s for s in test_subjects if s.startswith("Case_")])
    print("  CU  :", [s for s in test_subjects if s.lower().startswith("cu")])

# ===============================
# METRIC CONTAINERS
# ===============================
rmse_train_folds = []
rmse_test_folds  = []

all_preds_train, all_true_train = [], []
all_preds_test,  all_true_test  = [], []

feature_importances = np.zeros(len(features))

# ===============================
# TRAINING LOOP
# ===============================
for fold_num, (train_idx, test_idx) in enumerate(fold_indices, start=1):

    print(f"\n================ Fold {fold_num} ================")

    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]
    test_records = df.iloc[test_idx]["Record"].unique()

    # -------------------------------
    # Randomized Search (RF Regressor)
    # -------------------------------
    param_dist = {
        "n_estimators": [50, 100, 200, 300],
        "max_depth": [3, 5, 7],
        "min_samples_split": randint(2, 10),
        "min_samples_leaf": randint(1, 5),
    }

    base_model = RandomForestRegressor(
        random_state=random_state,
        n_jobs=-1,
    )

    search = RandomizedSearchCV(
        base_model,
        param_distributions=param_dist,
        n_iter=50,
        cv=3,
        scoring="neg_mean_squared_error",
        random_state=random_state,
        n_jobs=-1,
    )

    search.fit(X_train, y_train)
    model = search.best_estimator_

    # -------------------------------
    # Predictions
    # -------------------------------
    preds_train = medfilt(model.predict(X_train), kernel_size=5)
    preds_test  = medfilt(model.predict(X_test),  kernel_size=5)

    rmse_train = np.sqrt(mean_squared_error(y_train, preds_train))
    rmse_test  = np.sqrt(mean_squared_error(y_test, preds_test))

    print(f"RMSE (Train/Test): {rmse_train:.4f} / {rmse_test:.4f}")

    rmse_train_folds.append(rmse_train)
    rmse_test_folds.append(rmse_test)

    all_preds_train.extend(preds_train)
    all_true_train.extend(y_train)

    all_preds_test.extend(preds_test)
    all_true_test.extend(y_test)

    feature_importances += model.feature_importances_

    # ===============================
    # SUBJECT-LEVEL PLOTS (UNCHANGED)
    # ===============================
    for subject_id in test_records:

        sr = get_sampling_rate(subject_id)

        subject_df = df.iloc[test_idx]
        subject_df = subject_df[subject_df["Record"] == subject_id]
        subject_df = subject_df.sort_values("Start")

        X_sub = subject_df[features].values
        subject_df["Prediction"] = model.predict(X_sub)

        fig, axs = plt.subplots(2, 1, figsize=(14, 6), sharex=True)

        for _, row in subject_df.iterrows():
            ecg = row["ECG_Raw"]
            if isinstance(ecg, list) and len(ecg) > 0:
                t = np.linspace(
                    row["Start"] / sr,
                    (row["Start"] + 30 * sr) / sr,
                    len(ecg),
                )
                axs[0].plot(t, ecg, color="black", alpha=0.4)

        vtac_rows = subject_df[subject_df["VTAC_Label"] == 1]
        if not vtac_rows.empty:
            vtac_start = vtac_rows["Start"].min() / sr
            vtac_end   = vtac_rows["Start"].max() / sr
            axs[0].axvline(vtac_start, color="red", linestyle="--")
            axs[0].axvline(vtac_end,   color="red", linestyle="--")
            axs[0].axvline(vtac_start - sigmoid_seconds, color="blue", linestyle="--")

        axs[0].set_title(f"Raw ECG with VTAC Markers – {subject_id}")
        axs[0].set_ylabel("ECG")

        axs[1].plot(
            subject_df["Start"] / sr,
            subject_df["VTAC_Label_Regression"],
            label="True",
            color="blue",
            alpha=0.6,
        )
        axs[1].plot(
            subject_df["Start"] / sr,
            subject_df["Prediction"],
            label="Predicted",
            color="orange",
        )

        axs[1].legend()
        axs[1].set_ylabel("Regression Target")
        axs[1].set_xlabel("Time (s)")
        axs[1].set_ylim(0, 1.1)

        plt.tight_layout()
        plt.show()

# ===============================
# FEATURE IMPORTANCE
# ===============================
feature_importances /= n_folds
sorted_idx = np.argsort(feature_importances)[::-1]

plt.figure(figsize=(9, 5))
plt.bar(range(len(features)), feature_importances[sorted_idx])
plt.xticks(
    range(len(features)),
    [features[i] for i in sorted_idx],
    rotation=45,
    ha="right",
)
plt.title("Average Feature Importance (Random Forest Regressor)")
plt.ylabel("Importance")
plt.tight_layout()
plt.show()

# ===============================
# FINAL PERFORMANCE SUMMARY
# ===============================
rmse_train_folds = np.array(rmse_train_folds)
rmse_test_folds  = np.array(rmse_test_folds)

rmse_pooled_train = np.sqrt(
    mean_squared_error(all_true_train, all_preds_train)
)
rmse_pooled_test = np.sqrt(
    mean_squared_error(all_true_test, all_preds_test)
)

print("\n==============================")
print("CROSS-FOLD PERFORMANCE")
print("==============================")
print(f"Train RMSE: {rmse_train_folds.mean():.4f} ± {rmse_train_folds.std():.4f}")
print(f"Test  RMSE: {rmse_test_folds.mean():.4f} ± {rmse_test_folds.std():.4f}")

print("\n==============================")
print("POOLED PERFORMANCE")
print("==============================")
print(f"Pooled Train RMSE: {rmse_pooled_train:.4f}")
print(f"Pooled Test  RMSE: {rmse_pooled_test:.4f}")

### Train and Save Final VTAC Random Forest & XGBoost Regression Models

In [ ]:
# --- Train Final Random Forest Model on Full Data ---
rf_param_dist = {
    'n_estimators': [50, 100, 200, 300],
    'max_depth': [3, 5, 7],
    'min_samples_split': randint(2, 10),
    'min_samples_leaf': randint(1, 5)
}
rf_model = RandomForestRegressor(random_state=42)
rf_search = RandomizedSearchCV(rf_model, param_distributions=rf_param_dist, n_iter=50, cv=3,
                               scoring='neg_mean_squared_error', random_state=42, n_jobs=-1)
rf_search.fit(X, y)
rf_best = rf_search.best_estimator_

# Save model
model_path = os.path.join("model", "random_forest_vtac_model_regression.joblib")
joblib.dump(rf_best, model_path)
print(f"✅ Random Forest model saved to '{model_path}'")


xgb_param_dist = {
    'n_estimators': [50, 100, 200, 300],
    'max_depth': [3, 5, 7],
    'learning_rate': uniform(0.01, 0.2),
    'subsample': uniform(0.6, 0.4),
    'colsample_bytree': uniform(0.6, 0.4),
    'gamma': uniform(0, 5),
    'reg_alpha': uniform(0, 1),
    'reg_lambda': uniform(1, 2)
}
xgb_model = XGBRegressor(random_state=42, verbosity=0, eval_metric='rmse')
xgb_search = RandomizedSearchCV(xgb_model, param_distributions=xgb_param_dist, n_iter=20, cv=3,
                                scoring='neg_mean_squared_error', random_state=42, n_jobs=-1)
xgb_search.fit(X, y)
xgb_best = xgb_search.best_estimator_

# Save model
model_path = os.path.join("model", "xgboost_vtac_model_regression.joblib")
joblib.dump(xgb_best, model_path)
print(f"✅ XGBoost model saved to '{model_path}'")